# 11 - Clasificacion CT con backbone moderno

Este notebook anade un experimento final de clasificacion CT usando un backbone moderno. La idea no es cambiar todo el TFM, sino comprobar si una arquitectura mas reciente mejora la extraccion de caracteristicas frente a los modelos clasicos ya evaluados.

El experimento se centra en CT porque es la modalidad donde los resultados han sido mas dificiles. En CXR el rendimiento ya es muy alto, por lo que un nuevo backbone tendria menos valor experimental.

## Que se prueba en esta fase

Se entrena `ConvNeXt-Tiny` sobre la variante `top20_tissue` de CT. Esta variante conserva, por cada estudio, los 20 cortes con mayor contenido tisular. Asi se reduce el numero de slices poco informativos sin eliminar estudios completos.

La comparacion importante no es solo por slice. Como MosMedData etiqueta estudios completos, tambien se reconstruyen metricas por estudio agregando las probabilidades de los slices del mismo volumen.

## Preparacion del entorno

Esta celda localiza la raiz del proyecto, carga las librerias necesarias y define las rutas de resultados. Tambien usa el Python del entorno `.conda` del proyecto cuando esta disponible.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

NOTEBOOK_PATH = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_PATH.parent if NOTEBOOK_PATH.name == 'notebooks' else NOTEBOOK_PATH
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path('/Users/yuncaichen/Master/TFM:FINAL/TFM')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

PYTHON_BIN = PROJECT_ROOT / '.conda' / 'bin' / 'python'
PYTHON_BIN = str(PYTHON_BIN if PYTHON_BIN.exists() else sys.executable)

RESULTS_DIR = PROJECT_ROOT / 'results' / 'classification' / 'ct_informative_slices'
FIGURES_DIR = RESULTS_DIR / 'figures'

PROJECT_ROOT

## Preparar metadata de slices informativos

Esta celda genera o reutiliza la metadata de `top20_tissue`. No entrena todavia. Solo prepara la lista de imagenes CT que se usaran para el experimento moderno.

In [ ]:
subprocess.run(
    [PYTHON_BIN, str(PROJECT_ROOT / 'scripts' / 'run_ct_modern_backbone_experiments.py'), 'prepare'],
    cwd=PROJECT_ROOT,
    check=True,
)

## Entrenar ConvNeXt-Tiny en CT

Esta celda entrena `ConvNeXt-Tiny` con perdida ponderada (`weighted_ce`). El entrenamiento usa dos fases: primero se congela el backbone y se entrena solo la cabeza de clasificacion; despues se desbloquea todo el modelo para ajuste fino.

Si el entrenamiento ya existe, el script lo salta para no repetir trabajo.

Nota practica: por defecto se usan pesos preentrenados de ImageNet. Si el entorno no puede descargarlos, se puede repetir el comando anadiendo `--no-pretrained`, aunque metodologicamente es preferible usar transferencia de aprendizaje.

In [ ]:
RUN_TRAINING = True

if RUN_TRAINING:
    subprocess.run(
        [
            PYTHON_BIN,
            str(PROJECT_ROOT / 'scripts' / 'run_ct_modern_backbone_experiments.py'),
            'train-default',
            '--epochs', '20',
            '--head-epochs', '5',
            '--batch-size', '16',
            '--num-workers', '0',
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )

## Variantes de ajuste para reducir sobreajuste

El primer entrenamiento de ConvNeXt-Tiny puede aprender mejor el conjunto de entrenamiento que la validacion. Para comprobar si el problema es un fine-tuning demasiado agresivo, se proponen dos variantes adicionales.

`head_only` congela el backbone y entrena solo la cabeza de clasificacion. Esto prueba si las caracteristicas preentrenadas son suficientes sin modificar todo el extractor.

`gentle_ft` hace un ajuste fino mas suave: mas epocas con el backbone congelado, menor learning rate, mas weight decay y paciencia menor. Esto intenta mejorar generalizacion sin forzar demasiado el modelo.

In [ ]:
RUN_TUNING_VARIANTS = True

tuning_commands = [
    [
        PYTHON_BIN,
        str(PROJECT_ROOT / 'scripts' / 'run_ct_modern_backbone_experiments.py'),
        'train-one',
        '--variant', 'top20_tissue',
        '--architecture', 'convnext_tiny',
        '--strategy', 'weighted_ce',
        '--epochs', '8',
        '--head-epochs', '8',
        '--batch-size', '16',
        '--num-workers', '0',
        '--learning-rate', '0.0001',
        '--weight-decay', '0.0001',
        '--tag', 'head_only',
    ],
    [
        PYTHON_BIN,
        str(PROJECT_ROOT / 'scripts' / 'run_ct_modern_backbone_experiments.py'),
        'train-one',
        '--variant', 'top20_tissue',
        '--architecture', 'convnext_tiny',
        '--strategy', 'weighted_ce',
        '--epochs', '12',
        '--head-epochs', '8',
        '--batch-size', '16',
        '--num-workers', '0',
        '--learning-rate', '0.00005',
        '--weight-decay', '0.0001',
        '--early-stopping-patience', '3',
        '--tag', 'gentle_ft',
    ],
]

if RUN_TUNING_VARIANTS:
    for command in tuning_commands:
        subprocess.run(command, cwd=PROJECT_ROOT, check=True)

## Reconstruir analisis por slice y por estudio

Despues de entrenar, esta celda vuelve a construir las tablas de evaluacion de `ct_informative_slices`. Esto permite que el nuevo ConvNeXt-Tiny y sus variantes aparezcan junto a ResNet50 y DenseNet121 en las comparaciones por slice y por estudio.

In [ ]:
subprocess.run(
    [PYTHON_BIN, str(PROJECT_ROOT / 'scripts' / 'build_ct_informative_slice_analysis.py')],
    cwd=PROJECT_ROOT,
    check=True,
)

## Resultados principales

La primera tabla muestra la evaluacion por slice. La segunda muestra la evaluacion por estudio. Para CT, la evaluacion por estudio es especialmente importante porque la etiqueta original de MosMedData pertenece al volumen completo, no a cada slice individual.

In [ ]:
metrics_path = RESULTS_DIR / 'ct_informative_slice_metrics.csv'
metrics_df = pd.read_csv(metrics_path)

display(Markdown('### Metricas por slice'))
display(
    metrics_df[metrics_df['unit'].eq('slice')]
    .sort_values(['f1_macro', 'auc_roc_macro'], ascending=False)
    [['experiment', 'slice_selection_variant', 'unit', 'aggregation', 'n_samples', 'n_studies', 'accuracy', 'f1_macro', 'f1_weighted', 'auc_roc_macro']]
    .round(4)
)

display(Markdown('### Metricas por estudio'))
display(
    metrics_df[metrics_df['unit'].eq('study')]
    .sort_values(['f1_macro', 'auc_roc_macro'], ascending=False)
    [['experiment', 'slice_selection_variant', 'unit', 'aggregation', 'n_samples', 'n_studies', 'accuracy', 'f1_macro', 'f1_weighted', 'auc_roc_macro']]
    .round(4)
)

## Comparacion directa con la mejor referencia previa

Aqui comparamos el experimento moderno contra las referencias informativas ya entrenadas. Si ConvNeXt-Tiny mejora F1-macro o AUC por estudio, se puede defender como mejora. Si no mejora, sigue siendo un resultado util: indica que cambiar backbone no resuelve por si solo la dificultad de CT.

In [ ]:
comparison_experiments = [
    'ct_top20_tissue_convnext_tiny_weighted_ce',
    'ct_top20_tissue_head_only_convnext_tiny_weighted_ce',
    'ct_top20_tissue_gentle_ft_convnext_tiny_weighted_ce',
    'ct_top20_tissue_resnet50_weighted_ce',
    'ct_top16_tissue_resnet50_weighted_ce',
    'ct_top16_tissue_densenet121_baseline',
]

comparison_df = metrics_df[metrics_df['experiment'].isin(comparison_experiments)].copy()
display(
    comparison_df.sort_values(['unit', 'f1_macro', 'auc_roc_macro'], ascending=[True, False, False])
    [['experiment', 'unit', 'aggregation', 'accuracy', 'f1_macro', 'f1_weighted', 'auc_roc_macro']]
    .round(4)
)

## Figura resumen

Esta figura resume F1-macro de los experimentos con slices informativos. Sirve para ver rapidamente si el backbone moderno aporta mejora frente a las variantes anteriores.

In [ ]:
figure_path = FIGURES_DIR / 'ct_informative_slice_f1_macro.png'
if figure_path.exists():
    display(Image(filename=str(figure_path)))
else:
    display(Markdown(f'No se encontro la figura: `{figure_path}`'))

## Lectura para la memoria

Este experimento se puede documentar como una extension moderna de clasificacion CT. El objetivo es comprobar si una arquitectura reciente, ConvNeXt-Tiny, mejora la representacion de los cortes CT seleccionados. No sustituye a los experimentos principales; los complementa.

Si mejora los resultados por estudio, se puede presentar como evidencia de que una extraccion de caracteristicas mas actual ayuda en la tarea de severidad CT. Si no mejora, la conclusion tambien es defendible: la limitacion principal probablemente no esta solo en el backbone, sino en la naturaleza de MosMedData, el desbalanceo, las etiquetas a nivel de estudio y la conversion del volumen 3D a cortes 2D.